# 03 — Visualización y Análisis Exploratorio INE

**TFG: Spain's Migratory Flow**  
**Input:** `output/clean/ine_provincia_anio.csv`  
**Output:** Figuras en `output/03_visualizacion/ine/`

### Preguntas que responde este notebook
1. ¿Qué provincias tienen mayor / menor renta media? ¿Ha cambiado entre 2015-2023?
2. ¿Cómo se distribuye la desigualdad (Gini) en el territorio?
3. ¿Qué relación hay entre renta, Gini y estructura demográfica?
4. ¿Qué provincias son más dependientes de pensiones vs salario?
5. ¿Hay clusters de provincias con perfil socioeconómico similar?

Todas las figuras se guardan en alta resolución para incluir en la memoria del TFG.

In [1]:
# ── Librerías ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# ── Estilo de publicación ──────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.15)
plt.rcParams.update({
    'figure.dpi':        150,
    'savefig.dpi':       300,
    'savefig.bbox':      'tight',
    'axes.titlesize':    13,
    'axes.labelsize':    11,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'font.family':       'DejaVu Sans',
})

# Paleta personalizada para el TFG
PALETA_PROV = sns.color_palette('tab20', 52)
COL_RENTA   = '#1565C0'
COL_GINI    = '#C62828'
COL_DEMO    = '#2E7D32'
COL_ING     = '#E65100'

# ── Rutas ──────────────────────────────────────────────────────────────────────
INPUT_PATH = Path('../output/clean/ine_provincia_anio.csv')
FIG_DIR    = Path('../output/03_visualizacion/ine')
FIG_DIR.mkdir(parents=True, exist_ok=True)

def guardar_fig(nombre: str) -> None:
    """Guarda la figura actual en alta resolución con nombre descriptivo."""
    path = FIG_DIR / f'{nombre}.png'
    plt.savefig(path, dpi=300, bbox_inches='tight')
    print(f'  💾 Guardado: {path.name}')

print('✅ Configuración cargada')

✅ Configuración cargada


In [2]:
# ── Carga del dataset limpio ───────────────────────────────────────────────────
df = pd.read_csv(INPUT_PATH, encoding='utf-8')

print(f'✅ Dataset cargado: {df.shape}')
print(f'   Provincias: {df["cod_provincia"].nunique()}')
print(f'   Años: {sorted(df["year"].dropna().unique())}')
print(f'   Columnas: {list(df.columns)}')

# ── Identificar columnas por dominio ──────────────────────────────────────────
cols_renta  = [c for c in df.columns if c.startswith('renta__')]
cols_ing    = [c for c in df.columns if c.startswith('ingreso__')]
cols_desig  = [c for c in df.columns if c.startswith('desig__')]
cols_demog  = [c for c in df.columns if c.startswith('demog__')]

print(f'\n  Columnas renta    : {cols_renta}')
print(f'  Columnas ingresos : {cols_ing}')
print(f'  Columnas desig.   : {cols_desig}')
print(f'  Columnas demog.   : {cols_demog}')

# Identificar col_renta principal
col_renta_pp  = next((c for c in cols_renta if 'persona' in c and 'neta' in c), cols_renta[0] if cols_renta else None)
col_gini      = next((c for c in cols_desig if 'gini' in c), None)
col_edad_med  = next((c for c in cols_demog  if 'edad_media' in c), None)
col_pct_65    = next((c for c in cols_demog  if '65' in c), None)
col_pct_18    = next((c for c in cols_demog  if '18' in c), None)
col_salario   = next((c for c in cols_ing    if 'salario' in c), None)
col_pension   = next((c for c in cols_ing    if 'pension' in c), None)

print(f'\n  Indicadores principales identificados:')
for nombre, col in [('Renta/persona', col_renta_pp), ('Gini', col_gini),
                     ('Edad media', col_edad_med), ('% ≥65', col_pct_65),
                     ('Salario', col_salario), ('Pensiones', col_pension)]:
    print(f'    {nombre:<15}: {col}')

FileNotFoundError: [Errno 2] No such file or directory: '../output/clean/ine_provincia_anio.csv'

---
## FIG 1 — Ranking de renta neta media por persona (2023 vs 2015)

Comparamos el ranking entre el año inicial y el final para ver movilidad provincial.

In [ ]:
if col_renta_pp:
    df_2015 = df[df['year'] == 2015][['nombre_prov', col_renta_pp]].rename(columns={col_renta_pp: '2015'})
    df_2023 = df[df['year'] == 2023][['nombre_prov', col_renta_pp]].rename(columns={col_renta_pp: '2023'})
    df_rank  = df_2015.merge(df_2023, on='nombre_prov').dropna()
    df_rank['delta'] = df_rank['2023'] - df_rank['2015']
    df_rank = df_rank.sort_values('2023', ascending=True)
    
    fig, ax = plt.subplots(figsize=(10, 14))
    y = np.arange(len(df_rank))
    
    ax.barh(y, df_rank['2023'], height=0.5, color=COL_RENTA, alpha=0.85, label='2023')
    ax.scatter(df_rank['2015'], y, color='#90CAF9', zorder=5, s=40, label='2015')
    
    # Líneas de conexión 2015→2023
    for i, (_, row) in enumerate(df_rank.iterrows()):
        ax.plot([row['2015'], row['2023']], [i, i], color='gray', alpha=0.3, linewidth=1)
    
    ax.set_yticks(y)
    ax.set_yticklabels(df_rank['nombre_prov'], fontsize=9)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}€'))
    ax.set_xlabel('Renta neta media por persona (€)')
    ax.set_title('Renta neta media por persona: ranking provincial\n(mediana municipal · 2015 vs 2023)', pad=12)
    ax.legend(loc='lower right')
    
    plt.tight_layout()
    guardar_fig('01_ranking_renta_2015_2023')
    plt.show()

---
## FIG 2 — Evolución temporal de renta: provincias seleccionadas

Serie temporal de las 5 provincias con renta más alta y las 5 más baja (2023).

In [ ]:
if col_renta_pp:
    # Identificar top/bottom 5 por renta 2023
    df_23   = df[df['year'] == 2023][['nombre_prov', col_renta_pp]].dropna().sort_values(col_renta_pp)
    bot5    = df_23.head(5)['nombre_prov'].tolist()
    top5    = df_23.tail(5)['nombre_prov'].tolist()
    prov_sel = bot5 + top5
    
    df_sel = df[df['nombre_prov'].isin(prov_sel)][['nombre_prov', 'year', col_renta_pp]].dropna()
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    for i, prov in enumerate(prov_sel):
        sub   = df_sel[df_sel['nombre_prov'] == prov].sort_values('year')
        color = '#E53935' if prov in bot5 else '#1565C0'
        ls    = '--' if prov in bot5 else '-'
        ax.plot(sub['year'], sub[col_renta_pp], marker='o', linewidth=2,
                color=color, linestyle=ls, label=prov, alpha=0.85)
    
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}€'))
    ax.set_xlabel('Año')
    ax.set_ylabel('Renta neta media por persona (€)')
    ax.set_title('Evolución de la renta neta media: provincias extremas (2015–2023)', pad=12)
    ax.legend(title='Provincia', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
    
    # Anotación 2020 (COVID)
    ax.axvline(2020, color='gray', linestyle=':', alpha=0.6)
    ax.text(2020.1, ax.get_ylim()[0] * 1.02, 'COVID-19', fontsize=8, color='gray')
    
    plt.tight_layout()
    guardar_fig('02_evolucion_renta_top_bottom')
    plt.show()

---
## FIG 3 — Mapa de calor: Renta neta por provincia × año

In [ ]:
if col_renta_pp:
    pivot_renta = df.pivot_table(
        index='nombre_prov', columns='year', values=col_renta_pp
    ).dropna(how='all')
    pivot_renta = pivot_renta.sort_values(pivot_renta.columns[-1], ascending=False)
    
    fig, ax = plt.subplots(figsize=(13, 16))
    sns.heatmap(
        pivot_renta, annot=True, fmt='.0f', cmap='YlOrRd',
        linewidths=0.3,
        cbar_kws={'label': '€/persona (mediana municipal)'},
        ax=ax
    )
    ax.set_title('Renta neta media por persona — Provincia × Año', pad=12)
    ax.set_xlabel('Año')
    ax.set_ylabel('')
    plt.tight_layout()
    guardar_fig('03_heatmap_renta_prov_anio')
    plt.show()

---
## FIG 4 — Desigualdad: Gini por comunidad autónoma (2015 vs 2023)

Agrupamos las provincias por CCAA para dar contexto regional.

In [ ]:
# Mapa de provincia → CCAA
PROV_CCAA = {
    'Álava':'País Vasco','Gipuzkoa':'País Vasco','Bizkaia':'País Vasco',
    'Navarra':'Navarra','La Rioja':'La Rioja','Cantabria':'Cantabria',
    'Asturias':'Asturias','A Coruña':'Galicia','Lugo':'Galicia',
    'Ourense':'Galicia','Pontevedra':'Galicia',
    'Lleida':'Cataluña','Girona':'Cataluña','Barcelona':'Cataluña','Tarragona':'Cataluña',
    'Alicante':'C. Valenciana','Castellón':'C. Valenciana','Valencia':'C. Valenciana',
    'Baleares':'Baleares','Las Palmas':'Canarias','S.C. Tenerife':'Canarias',
    'Sevilla':'Andalucía','Cádiz':'Andalucía','Huelva':'Andalucía','Málaga':'Andalucía',
    'Granada':'Andalucía','Jaén':'Andalucía','Córdoba':'Andalucía','Almería':'Andalucía',
    'Badajoz':'Extremadura','Cáceres':'Extremadura',
    'Madrid':'Madrid','Toledo':'C.-La Mancha','Ciudad Real':'C.-La Mancha',
    'Cuenca':'C.-La Mancha','Albacete':'C.-La Mancha','Guadalajara':'C.-La Mancha',
    'Burgos':'Castilla y León','León':'Castilla y León','Salamanca':'Castilla y León',
    'Valladolid':'Castilla y León','Zamora':'Castilla y León','Ávila':'Castilla y León',
    'Segovia':'Castilla y León','Soria':'Castilla y León','Palencia':'Castilla y León',
    'Huesca':'Aragón','Teruel':'Aragón','Zaragoza':'Aragón',
    'Murcia':'Murcia','Ceuta':'Ceuta','Melilla':'Melilla'
}

if col_gini:
    df_g = df[df['year'].isin([2015, 2023])][['nombre_prov', 'year', col_gini]].dropna()
    df_g['ccaa'] = df_g['nombre_prov'].map(PROV_CCAA).fillna('Otras')
    
    df_ccaa = df_g.groupby(['ccaa', 'year'])[col_gini].median().reset_index()
    pivot_ccaa = df_ccaa.pivot(index='ccaa', columns='year', values=col_gini).dropna()
    pivot_ccaa = pivot_ccaa.sort_values(2023, ascending=True)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    y = np.arange(len(pivot_ccaa))
    
    ax.barh(y - 0.2, pivot_ccaa[2015], height=0.35, color='#90CAF9', label='2015', alpha=0.9)
    ax.barh(y + 0.2, pivot_ccaa[2023], height=0.35, color=COL_GINI,  label='2023', alpha=0.9)
    ax.set_yticks(y)
    ax.set_yticklabels(pivot_ccaa.index)
    ax.set_xlabel('Índice de Gini (mediana provincial)')
    ax.set_title('Índice de Gini por Comunidad Autónoma\n(mediana provincial, 2015 vs 2023)', pad=12)
    ax.legend()
    ax.axvline(30, color='gray', linestyle='--', alpha=0.5)
    ax.text(30.2, ax.get_ylim()[1] * 0.98, 'Gini=30', fontsize=8, color='gray')
    
    plt.tight_layout()
    guardar_fig('04_gini_por_ccaa')
    plt.show()

---
## FIG 5 — Renta vs Gini (2023): ¿las provincias más ricas son más desiguales?

Scatter plot con regresión lineal. Pregunta clave para el análisis migratorio.

In [ ]:
if col_renta_pp and col_gini:
    df_23 = df[df['year'] == 2023][['nombre_prov', col_renta_pp, col_gini]].dropna()
    
    fig, ax = plt.subplots(figsize=(11, 8))
    
    # Scatter
    scatter = ax.scatter(
        df_23[col_renta_pp], df_23[col_gini],
        c=df_23[col_renta_pp], cmap='RdYlGn', s=80, alpha=0.85,
        edgecolors='white', linewidths=0.5
    )
    plt.colorbar(scatter, ax=ax, label='Renta neta media por persona (€)')
    
    # Regresión
    from numpy.polynomial import polynomial as P
    x = df_23[col_renta_pp].values
    y = df_23[col_gini].values
    mask = ~(np.isnan(x) | np.isnan(y))
    coef = np.polyfit(x[mask], y[mask], 1)
    x_line = np.linspace(x[mask].min(), x[mask].max(), 100)
    ax.plot(x_line, np.polyval(coef, x_line), color='#B71C1C', linewidth=2,
            linestyle='--', label=f'Regresión lineal (pendiente={coef[0]:.4f})')
    
    # Etiquetas de provincias notables
    for _, row in df_23.iterrows():
        if (row[col_renta_pp] > df_23[col_renta_pp].quantile(0.85) or
            row[col_renta_pp] < df_23[col_renta_pp].quantile(0.15) or
            row[col_gini] > df_23[col_gini].quantile(0.85) or
            row[col_gini] < df_23[col_gini].quantile(0.15)):
            ax.annotate(row['nombre_prov'],
                        (row[col_renta_pp], row[col_gini]),
                        xytext=(5, 3), textcoords='offset points', fontsize=8)
    
    ax.set_xlabel('Renta neta media por persona (€)')
    ax.set_ylabel('Índice de Gini')
    ax.set_title('Renta vs Desigualdad (Gini) por provincia — 2023', pad=12)
    ax.legend()
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}€'))
    
    plt.tight_layout()
    guardar_fig('05_scatter_renta_vs_gini')
    plt.show()
    
    # Correlación
    corr = df_23[[col_renta_pp, col_gini]].corr().iloc[0, 1]
    print(f'📊 Correlación Pearson renta vs Gini (2023): r = {corr:.3f}')
    interpretacion = 'correlación negativa moderada' if corr < -0.3 else ('positiva' if corr > 0.3 else 'baja')
    print(f'   → {interpretacion}')

---
## FIG 6 — Envejecimiento vs Renta

¿Las provincias más envejecidas tienen menor renta? (relevante para migración).

In [ ]:
if col_renta_pp and col_edad_med and col_pct_65:
    df_23 = df[df['year'] == 2023][['nombre_prov', col_renta_pp, col_edad_med, col_pct_65]].dropna()
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    
    for ax, col_x, xlabel in [
        (axes[0], col_edad_med, 'Edad media de la población'),
        (axes[1], col_pct_65, '% población ≥65 años'),
    ]:
        sub = df_23[['nombre_prov', col_renta_pp, col_x]].dropna()
        ax.scatter(sub[col_x], sub[col_renta_pp], color=COL_DEMO, s=70, alpha=0.8,
                   edgecolors='white')
        
        # Regresión
        coef = np.polyfit(sub[col_x].values, sub[col_renta_pp].values, 1)
        x_r  = np.linspace(sub[col_x].min(), sub[col_x].max(), 100)
        ax.plot(x_r, np.polyval(coef, x_r), '--', color='#1B5E20', linewidth=1.8)
        
        # Etiquetas
        for _, row in sub.iterrows():
            if (row[col_renta_pp] > sub[col_renta_pp].quantile(0.88) or
                row[col_x] > sub[col_x].quantile(0.88)):
                ax.annotate(row['nombre_prov'], (row[col_x], row[col_renta_pp]),
                            xytext=(3, 3), textcoords='offset points', fontsize=7.5)
        
        corr = sub[[col_x, col_renta_pp]].corr().iloc[0, 1]
        ax.set_xlabel(xlabel)
        ax.set_ylabel('Renta neta media por persona (€)')
        ax.set_title(f'{xlabel}\nvs Renta — r={corr:.2f}', fontsize=11)
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}€'))
    
    plt.suptitle('Envejecimiento vs Renta por provincia (2023)', fontsize=13)
    plt.tight_layout()
    guardar_fig('06_envejecimiento_vs_renta')
    plt.show()

---
## FIG 7 — Fuentes de ingreso: salario vs pensiones por provincia

Stacked bar con las 5 fuentes de ingreso para todas las provincias (2023).

In [ ]:
if cols_ing:
    df_23 = df[df['year'] == 2023][['nombre_prov'] + cols_ing].dropna(subset=[cols_ing[0]])
    df_23 = df_23.set_index('nombre_prov')
    
    # Ordenar por % salario descendente
    col_sal_local = next((c for c in cols_ing if 'salario' in c), None)
    if col_sal_local:
        df_23 = df_23.sort_values(col_sal_local, ascending=True)
    
    # Nombres cortos para columnas
    rename_ing = {c: c.replace('ingreso__fuente_de_ingreso_', '').replace('_', ' ').title()
                  for c in cols_ing}
    df_23 = df_23.rename(columns=rename_ing)
    
    fig, ax = plt.subplots(figsize=(12, 14))
    colors = ['#1565C0', '#E65100', '#C62828', '#2E7D32', '#6A1B9A']
    df_23.plot(kind='barh', stacked=True, ax=ax, color=colors[:len(df_23.columns)], 
               width=0.75, alpha=0.88)
    
    ax.set_xlabel('Porcentaje (%)')
    ax.set_title('Distribución de fuentes de ingreso por provincia (2023)\n(mediana municipal)', pad=12)
    ax.legend(title='Fuente', bbox_to_anchor=(1.01, 1), loc='upper left')
    ax.set_xlim(0, 105)
    
    plt.tight_layout()
    guardar_fig('07_fuentes_ingreso_provincia')
    plt.show()

---
## FIG 8 — Matriz de correlación de todos los indicadores INE (2023)

Heatmap de correlación de Pearson entre todos los indicadores. Fundamental para guiar la regresión.

In [ ]:
cols_num = [c for c in df.columns if c not in ['cod_provincia', 'nombre_prov', 'year', 'n_municipios_con_dato']]

df_23_num = df[df['year'] == 2023][cols_num].dropna(axis=1, thresh=30)
corr_matrix = df_23_num.corr(method='pearson')

# Nombres cortos para visualización
def abrev(col: str) -> str:
    return (
        col.replace('renta__', 'R:').replace('ingreso__', 'I:').replace('desig__', 'D:')
        .replace('demog__', 'Dm:').replace('fuente_de_ingreso_', '')
        .replace('_', ' ').replace('indicadores de renta media', '')
        [:35]
    )

corr_matrix.index   = [abrev(c) for c in corr_matrix.index]
corr_matrix.columns = [abrev(c) for c in corr_matrix.columns]

fig, ax = plt.subplots(figsize=(16, 14))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))   # solo triángulo inferior
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
    vmin=-1, vmax=1, center=0, linewidths=0.3, square=True,
    cbar_kws={'label': 'Correlación de Pearson (r)'},
    annot_kws={'size': 8}, ax=ax
)
ax.set_title('Matriz de correlación — Indicadores INE (provincia × año, 2023)', pad=12)
plt.tight_layout()
guardar_fig('08_matriz_correlacion_ine')
plt.show()

# Top 10 correlaciones (excluyendo self-correlación)
corr_vals = corr_matrix.unstack()
corr_vals = corr_vals[corr_vals.index.get_level_values(0) != corr_vals.index.get_level_values(1)]
corr_vals = corr_vals.abs().sort_values(ascending=False).drop_duplicates()
print('\n📊 Top 10 correlaciones más fuertes (2023):')
print(corr_vals.head(10).to_string())

---
## FIG 9 — Análisis de componentes: ¿qué indicadores explican más varianza?

PCA sobre los indicadores INE para detectar las dimensiones latentes del territorio.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Preparar datos: 2023, sin NaN
df_pca = df[df['year'] == 2023][['nombre_prov'] + cols_num].dropna(axis=1, thresh=40).dropna()
X = df_pca[cols_num].select_dtypes(include='number')
X_cols = X.columns.tolist()

if len(X_cols) >= 3 and len(df_pca) >= 10:
    scaler = StandardScaler()
    X_sc   = scaler.fit_transform(X.fillna(X.median()))
    
    pca = PCA()
    comps = pca.fit_transform(X_sc)
    
    # Varianza explicada
    var_exp = pca.explained_variance_ratio_ * 100
    var_cum = np.cumsum(var_exp)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Scree plot
    n_show = min(10, len(var_exp))
    axes[0].bar(range(1, n_show + 1), var_exp[:n_show], color='#1565C0', alpha=0.8)
    ax2 = axes[0].twinx()
    ax2.plot(range(1, n_show + 1), var_cum[:n_show], 'o-', color='#E53935', linewidth=2)
    ax2.axhline(80, color='gray', linestyle='--', alpha=0.5)
    ax2.set_ylabel('% varianza acumulada', color='#E53935')
    axes[0].set_xlabel('Componente')
    axes[0].set_ylabel('% varianza explicada')
    axes[0].set_title('Scree plot — PCA indicadores INE')
    
    # Biplot PC1 vs PC2
    axes[1].scatter(comps[:, 0], comps[:, 1], s=60, alpha=0.8, color='#1565C0')
    for i, prov in enumerate(df_pca['nombre_prov'].values):
        axes[1].annotate(prov, (comps[i, 0], comps[i, 1]),
                         xytext=(3, 3), textcoords='offset points', fontsize=7)
    axes[1].set_xlabel(f'PC1 ({var_exp[0]:.1f}% varianza)')
    axes[1].set_ylabel(f'PC2 ({var_exp[1]:.1f}% varianza)')
    axes[1].set_title('Provincias en espacio PCA — Indicadores INE 2023')
    axes[1].axhline(0, color='gray', linewidth=0.8)
    axes[1].axvline(0, color='gray', linewidth=0.8)
    
    plt.tight_layout()
    guardar_fig('09_pca_indicadores_ine')
    plt.show()
    
    # Loadings de PC1 (qué variables contribuyen más)
    loadings = pd.Series(pca.components_[0], index=X_cols).abs().sort_values(ascending=False)
    print('\n📊 Variables con mayor peso en PC1 (primer componente):')
    print(loadings.head(8).to_string())
else:
    print('⚠️  Datos insuficientes para PCA (necesita ≥3 variables y ≥10 provincias sin NaN)')

---
## FIG 10 — Resumen estadístico descriptivo (tabla de publicación)

Tabla lista para incluir en la memoria del TFG.

In [ ]:
# Estadísticas descriptivas del panel completo (todas las provincias y años)
df_num = df[cols_num].select_dtypes(include='number')

stats = df_num.describe().T[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']]
stats['cv%'] = (stats['std'] / stats['mean'].abs() * 100).round(1)  # coef. variación
stats = stats.round(2)
stats.index = [abrev(c) for c in stats.index]

print('📋 ESTADÍSTICAS DESCRIPTIVAS — Panel provincia × año (2015–2023)')
print('='*90)
print(stats.to_string())

# Guardar como CSV para incluir en la memoria
stats.to_csv(FIG_DIR / '../ine_estadisticas_descriptivas.csv', encoding='utf-8')
print('\n✅ Tabla guardada en output/03_visualizacion/ine_estadisticas_descriptivas.csv')

print(f'\n✅ Todas las figuras guardadas en: {FIG_DIR}')
print(f'   Figuras generadas: {len(list(FIG_DIR.glob("*.png")))}')